# Compare Predictions: R sommer vs py-sommer

This notebook fits the same random-intercept mixed model in both implementations and compares per-observation predictions.

Model:
- $y = 2 + u_{id} + e$
- $u_{id} \sim N(0, \sigma_u^2)$
- $e \sim N(0, \sigma_e^2)$

## Environment Setup (run once)

Use the project environment for this notebook so `pysommer` imports correctly.

From a terminal in this repository root:

```bash
uv sync --dev
uv run jupyter kernelspec remove py-sommer -f || true
uv run python -m ipykernel install --user --name py-sommer --display-name "Python (py-sommer)"
uv run jupyter lab
```

Then select kernel **Python (py-sommer)** in this notebook before running the code cells.

If you previously used `uv run --with ...` for kernel install, these commands replace the broken temporary kernel path with a stable one.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import tempfile

import numpy as np

from pysommer import mmes

ROOT = Path.cwd()
print(f'Workspace: {ROOT}')

Workspace: /Users/nico/Desktop/Projects/py-sommer/notebooks


In [2]:
# Generate deterministic synthetic data used by both implementations.
rng = np.random.default_rng(123)
n_groups = 20
reps = 2
group = np.repeat(np.arange(n_groups), reps)
n = group.size

X = np.ones((n, 1), dtype=float)
Z = np.eye(n_groups)[group]

u_true = rng.normal(0.0, np.sqrt(1.2), size=(n_groups, 1))
e = rng.normal(0.0, np.sqrt(0.4), size=(n, 1))
y = 2.0 + Z @ u_true + e

print('n observations:', n)
print('n groups:', n_groups)

n observations: 40
n groups: 20


In [3]:
# Fit py-sommer and build full predictions (fixed + random effects).
fit_py = mmes(Y=y, X=X, Z=[Z], K=[np.eye(n_groups)], iters=50)
beta_py = fit_py['beta']
u_py = fit_py['u'][0]
yhat_py = X @ beta_py + Z @ u_py

print('Python converged:', fit_py['converged'])
print('Python theta:', np.asarray(fit_py['theta']).round(6))

Python converged: True
Python theta: [0.829503 0.42071 ]


In [4]:
# Run an R script through Rscript to fit sommer::mmes on the same data and export predictions to JSON.
rscript_path = shutil.which('Rscript')
if rscript_path is None:
    raise RuntimeError('Rscript not found on PATH. Install R and ensure Rscript is available.')

with tempfile.TemporaryDirectory() as td:
    td_path = Path(td)
    data_csv = td_path / 'model_data.csv'
    r_file = td_path / 'run_sommer_compare.R'
    out_json = td_path / 'r_results.json'

    # Save data for R.
    data_matrix = np.column_stack([group + 1, y.reshape(-1)])
    np.savetxt(data_csv, data_matrix, delimiter=',', header='id,y', comments='')

    # Use a plain template string so R braces are not parsed by Python f-strings.
    r_template = '''
suppressPackageStartupMessages(library(sommer))
suppressPackageStartupMessages(library(jsonlite))

dat <- read.csv("__DATA_CSV__")
dat$id <- as.factor(dat$id)

fit <- mmes(
  y ~ 1,
  random = ~id,
  rcov = ~units,
  data = dat,
  nIters = 50,
  tolParConvNorm = 1e-6,
  tolParConvLL = 1e-6,
  verbose = FALSE,
  dateWarning = FALSE
)

pred <- tryCatch(as.numeric(fitted(fit)), error = function(e) NULL)
if (is.null(pred)) {
  b0 <- if (!is.null(fit$Beta)) as.numeric(fit$Beta)[1] else mean(dat$y)
  u <- NULL
  if (!is.null(fit$u)) {
    u <- as.numeric(fit$u)
  }
  if (is.null(u) && !is.null(fit$U)) {
    u <- as.numeric(fit$U[, 1])
  }
  if (is.null(u)) {
    pred <- rep(b0, nrow(dat))
  } else {
    idx <- as.integer(dat$id)
    if (length(u) < max(idx)) {
      u <- c(u, rep(0, max(idx) - length(u)))
    }
    pred <- b0 + u[idx]
  }
}

sigma <- if (!is.null(fit$sigma)) as.numeric(fit$sigma) else NA
write_json(list(yhat = pred, sigma = sigma), path = "__OUT_JSON__", auto_unbox = TRUE)
'''

    r_code = (
        r_template
        .replace('__DATA_CSV__', data_csv.as_posix())
        .replace('__OUT_JSON__', out_json.as_posix())
    )

    r_file.write_text(r_code)

    proc = subprocess.run(
        [rscript_path, str(r_file)],
        capture_output=True,
        text=True,
        check=False,
    )

    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError('R sommer run failed. Ensure sommer and jsonlite are installed in R.')

    payload = json.loads(out_json.read_text())

yhat_r = np.asarray(payload['yhat'], dtype=float).reshape(-1, 1)
theta_r = np.asarray(payload.get('sigma', []), dtype=float)

print('R predictions loaded:', yhat_r.shape[0])
print('R sigma:', theta_r)

R predictions loaded: 40
R sigma: nan


In [5]:
# Compare prediction vectors.
if yhat_r.shape != yhat_py.shape:
    raise ValueError(f'Shape mismatch: R={yhat_r.shape}, Python={yhat_py.shape}')

diff = yhat_py - yhat_r
mae = float(np.mean(np.abs(diff)))
rmse = float(np.sqrt(np.mean(diff**2)))
corr = float(np.corrcoef(yhat_py.ravel(), yhat_r.ravel())[0, 1])
max_abs = float(np.max(np.abs(diff)))

print('Prediction agreement metrics')
print('  MAE    :', round(mae, 8))
print('  RMSE   :', round(rmse, 8))
print('  Corr   :', round(corr, 8))
print('  MaxAbs :', round(max_abs, 8))

Prediction agreement metrics
  MAE    : 2.377e-05
  RMSE   : 2.83e-05
  Corr   : 1.0
  MaxAbs : 4.821e-05


In [6]:
# Display a quick side-by-side sample.
k = 10
rows = np.column_stack([
    np.arange(1, k + 1),
    y[:k, 0],
    yhat_py[:k, 0],
    yhat_r[:k, 0],
    (yhat_py[:k, 0] - yhat_r[:k, 0]),
])

header = 'row, y_obs, yhat_py, yhat_r, py_minus_r'
print(header)
for row in rows:
    print(', '.join(f'{v:.6f}' for v in row))

row, y_obs, yhat_py, yhat_r, py_minus_r
1.000000, -0.479655, 0.821846, 0.821800, 0.000046
2.000000, 1.440095, 0.821846, 0.821800, 0.000046
3.000000, 2.572123, 2.385931, 2.385900, 0.000031
4.000000, 2.309765, 2.385931, 2.385900, 0.000031
5.000000, 3.888210, 3.313207, 3.313200, 0.000007
6.000000, 3.318527, 3.313207, 3.313200, 0.000007
7.000000, 3.023234, 2.797992, 2.798000, -0.000008
8.000000, 2.891765, 2.797992, 2.798000, -0.000008
9.000000, 3.256378, 2.938658, 2.938700, -0.000042
10.000000, 3.011297, 2.938658, 2.938700, -0.000042


## Example: Set Up a Model in Python

This is a minimal template showing how to build `Y`, `X`, `Z`, and `K` for `pysommer.mmes` using explicit matrices.

In [7]:
# Minimal model setup example with explicit matrices.

import numpy as np
from pysommer import mmes

n_groups_ex = 12
reps_ex = 4
group_ex = np.repeat(np.arange(n_groups_ex), reps_ex)
n_ex = group_ex.size

# Fixed-effects design matrix: intercept only
X_ex = np.ones((n_ex, 1), dtype=float)

# Random-effects design matrix for group IDs
Z_ex = np.eye(n_groups_ex)[group_ex]

# Covariance/relationship matrix for the random term
K_ex = np.eye(n_groups_ex, dtype=float)

# Build a synthetic response for demonstration
rng_ex = np.random.default_rng(321)
u_ex = rng_ex.normal(0.0, np.sqrt(1.0), size=(n_groups_ex, 1))
e_ex = rng_ex.normal(0.0, np.sqrt(0.5), size=(n_ex, 1))
y_ex = 1.5 + Z_ex @ u_ex + e_ex

fit_ex = mmes(
    Y=y_ex,
    X=X_ex,
    Z=[Z_ex],
    K=[K_ex],
    iters=40,
)

yhat_ex = X_ex @ fit_ex["beta"] + Z_ex @ fit_ex["u"][0]

print("converged:", fit_ex["converged"])
print("theta:", np.asarray(fit_ex["theta"]).round(6))
print("beta:", np.asarray(fit_ex["beta"]).ravel().round(6))
print("first 5 predictions:", yhat_ex[:5, 0].round(6))

converged: True
theta: [0.552061 0.445856]
beta: [1.225082]
first 5 predictions: [0.98168 0.98168 0.98168 0.98168 1.70589]


## Notes

- This compares predictions for a simple random-intercept model and is intended as a reproducible cross-language sanity check.
- If R output differs across sommer versions, keep the same data seed and compare relative agreement (RMSE/correlation) rather than exact equality.